# SmartFit AI: Workout Plan Generator

### **Goal:** To create a rule-based engine that generates a personalized weekly workout plan.

**This notebook will perform the following steps:**
1.  **Load and Clean:** Load the `mega_gym_dataset.csv` and preprocess it for our use case.
2.  **Define Logic:** Establish rules for workout plan creation based on goals like "muscle gain" or "fat loss".
3.  **Build the Generator:** Create a single function that takes user inputs (level, goal, equipment) and returns a structured workout plan.
4.  **Test the Generator:** Run the function with sample inputs to verify the output.

This function will be the core "AI" logic that you integrate into your FastAPI backend.

In [ ]:
# @title Step 1: Setup and Data Loading
import pandas as pd
import json

# --- IMPORTANT ---
# Upload your 'mega_gym_dataset.csv' file using the file browser on the left.
file_path = '/content/drive/MyDrive/Colab Notebooks/01_megaGymDataset.csv'

try:
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully!")
except FileNotFoundError:
    print(f"Error: Make sure '{file_path}' is uploaded to your Colab session.")
    df = None # Set df to None if file is not found to avoid further errors

Dataset loaded successfully!


In [ ]:
df.head(
)

,Unnamed: 0,Title,Desc,Type,BodyPart,Equipment,Level,Rating,RatingDesc
0,0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0,NaN
1,1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
2,2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
3,3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
4,4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,NaN,NaN


### Step 2: Data Exploration and Preprocessing

Before we can generate plans, we need to understand our data and clean it up. We will:
- Check for missing values.
- Standardize the `Level` column.
- Drop columns we don't need.
- See the unique values for `BodyPart`, `Equipment`, and `Level` to understand our filtering options.

In [ ]:
# @title Data Cleaning
if df is not None:
    # --- Step 1: Inspect data ---
    print("Original DataFrame Info:")
    df.info()
    print("\nMissing values before cleaning:\n", df.isnull().sum())

    # --- Step 2: Clean data ---

    # Drop unnecessary columns
    df_cleaned = df.drop(columns=['Unnamed: 0', 'Rating', 'RatingDesc', 'Desc'])

    # Fill missing Equipment with 'Body Weight'
    df_cleaned['Equipment'] = df_cleaned['Equipment'].fillna('Body Weight')

    # Standardize Level values (Expert → Intermediate)
    level_mapping = {
        'Beginner': 'Beginner',
        'Intermediate': 'Intermediate',
        'Expert': 'Intermediate'
    }
    df_cleaned['Level'] = df_cleaned['Level'].map(level_mapping)

    # Drop rows with missing essential info
    df_cleaned.dropna(subset=['Level', 'BodyPart'], inplace=True)

    # --- Step 3: Verify ---
    print("\nCleaned DataFrame Info:")
    df_cleaned.info()
    print("\nMissing values after cleaning:\n", df_cleaned.isnull().sum())

    print("\n--- Unique Filterable Values ---")
    print("\nBody Parts:\n", df_cleaned['BodyPart'].unique())
    print("\nEquipment:\n", df_cleaned['Equipment'].unique())
    print("\nLevels:\n", df_cleaned['Level'].unique())


Original DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2918 entries, 0 to 2917
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  2918 non-null   int64  
 1   Title       2918 non-null   object 
 2   Desc        1368 non-null   object 
 3   Type        2918 non-null   object 
 4   BodyPart    2918 non-null   object 
 5   Equipment   2886 non-null   object 
 6   Level       2918 non-null   object 
 7   Rating      1031 non-null   float64
 8   RatingDesc  862 non-null    object 
dtypes: float64(1), int64(1), object(7)
memory usage: 205.3+ KB

Missing values before cleaning:
 Unnamed: 0       0
Title            0
Desc          1550
Type             0
BodyPart         0
Equipment       32
Level            0
Rating        1887
RatingDesc    2056
dtype: int64

Cleaned DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2918 entries, 0 to 2917
Data columns (total 5 columns):
 #   

### Step 3: Define Workout Logic and Rules

This is the core of our "AI". We'll define how to construct a workout plan. We'll use a common Push/Pull/Legs split for muscle gain and a full-body routine for fat loss.

In [ ]:
# @title Workout Rules Definition

# Define which body parts belong to which day in a Push/Pull/Legs (PPL) split
PPL_split = {
    'Push': [
        'Chest', 'Shoulders', 'Triceps'
    ],
    'Pull': [
        'Lats', 'Middle Back', 'Lower Back', 'Biceps', 'Forearms', 'Traps'
    ],
    'Legs': [
        'Quadriceps', 'Hamstrings', 'Glutes', 'Calves', 'Adductors', 'Abductors'
    ],
    'Core': [
        'Abdominals'
    ]
}

# Define sets and reps based on the goal
sets_reps_schema = {
    'muscle_gain': '3-4 sets of 8-12 reps',
    'fat_loss': '3 sets of 15-20 reps',
    'strength_gain': '4-5 sets of 3-6 reps'
}


### Step 4: Build the Generator Function

Now, we wrap all our logic into a single, reusable function. This function will be called by our FastAPI endpoint. It will filter the dataframe and randomly select exercises based on the defined rules.

In [ ]:
# @title The Workout Plan Generator Function
def generate_workout_plan(df, user_level, user_goal, available_equipment):
    """
    Generates a 3–4 day workout plan based on user preferences.

    Args:
        df (pd.DataFrame): The cleaned exercises dataframe.
        user_level (str): 'Beginner' or 'Intermediate'.
        user_goal (str): 'muscle_gain', 'fat_loss', or 'strength_gain'.
        available_equipment (list): A list of equipment the user has.

    Returns:
        dict: A dictionary representing the weekly workout plan.
    """
    # Filter exercises by user's level and available equipment
    filtered_df = df[
        (df['Level'] == user_level) &
        (df['Equipment'].isin(available_equipment))
    ].copy()

    if filtered_df.empty:
        return {
            "Error": "No exercises found for your selected level and equipment. Please broaden your criteria."
        }

    workout_plan = {}
    reps_sets = sets_reps_schema.get(user_goal, '3 sets of 10–12 reps')

    # PPL split or full-body depending on goal
    if user_goal in ['muscle_gain', 'strength_gain']:
        # Push / Pull / Legs / Core split
        for day, body_parts in PPL_split.items():
            day_exercises = []

            for part in body_parts:
                part_exercises = filtered_df[filtered_df['BodyPart'] == part]
                num_samples = min(2, len(part_exercises))
                if num_samples > 0:
                    selected = part_exercises.sample(n=num_samples)
                    for _, row in selected.iterrows():
                        day_exercises.append({
                            "Exercise": row['Title'],   # <-- FIXED HERE
                            "BodyPart": row['BodyPart'],
                            "SetsReps": reps_sets
                        })

            if day_exercises:
                workout_plan[f"Day {len(workout_plan) + 1}: {day}"] = day_exercises

    elif user_goal == 'fat_loss':
        # Full-body fat loss workouts
        # Updated compound body parts to match your dataset
        compound_body_parts = [
            'Quadriceps', 'Hamstrings', 'Lats', 'Middle Back', 'Chest', 'Shoulders'
        ]

        for i in range(1, 4):  # 3 full-body days
            day_exercises = []
            for part in compound_body_parts:
                part_exercises = filtered_df[filtered_df['BodyPart'] == part]
                num_samples = min(1, len(part_exercises))
                if num_samples > 0:
                    selected = part_exercises.sample(n=num_samples)
                    for _, row in selected.iterrows():
                        day_exercises.append({
                            "Exercise": row['Title'],   # <-- FIXED HERE
                            "BodyPart": row['BodyPart'],
                            "SetsReps": reps_sets
                        })

            if day_exercises:
                workout_plan[f"Day {i}: Full Body"] = day_exercises

    return workout_plan


### Step 5: Test the Generator

Let's test our function with some sample inputs to see what a generated plan looks like.

In [ ]:
# @title Example Usage
import json

if 'df_cleaned' in locals() and df_cleaned is not None:
    # --- TEST CASE 1: Beginner, Muscle Gain, Body Weight Only ---
    print("--- TEST CASE 1: Beginner, Muscle Gain, Body Weight Only ---")
    user_profile_1 = {
        'level': 'Beginner',
        'goal': 'muscle_gain',
        'equipment': ['Body Weight']
    }
    plan_1 = generate_workout_plan(
        df_cleaned,
        user_profile_1['level'],
        user_profile_1['goal'],
        user_profile_1['equipment']
    )
    print(json.dumps(plan_1, indent=2))

    # --- TEST CASE 2: Intermediate, Fat Loss, with Dumbbells and Barbell ---
    print("\n\n--- TEST CASE 2: Intermediate, Fat Loss, with Dumbbells & Barbell ---")
    user_profile_2 = {
        'level': 'Intermediate',
        'goal': 'fat_loss',
        'equipment': ['Dumbbell', 'Barbell', 'Body Weight']
    }
    plan_2 = generate_workout_plan(
        df_cleaned,
        user_profile_2['level'],
        user_profile_2['goal'],
        user_profile_2['equipment']
    )
    print(json.dumps(plan_2, indent=2))
else:
    print("Error: df_cleaned not found or is None.")


--- TEST CASE 1: Beginner, Muscle Gain, Body Weight Only ---
{
  "Day 1: Push": [
    {
      "Exercise": "Kneeling Arm Drill",
      "BodyPart": "Shoulders",
      "SetsReps": "3-4 sets of 8-12 reps"
    },
    {
      "Exercise": "Elbow Circles",
      "BodyPart": "Shoulders",
      "SetsReps": "3-4 sets of 8-12 reps"
    }
  ],
  "Day 2: Pull": [
    {
      "Exercise": "Side-Lying Floor Stretch",
      "BodyPart": "Lats",
      "SetsReps": "3-4 sets of 8-12 reps"
    },
    {
      "Exercise": "Dynamic Back Stretch",
      "BodyPart": "Lats",
      "SetsReps": "3-4 sets of 8-12 reps"
    },
    {
      "Exercise": "Standing Pelvic Tilt",
      "BodyPart": "Lower Back",
      "SetsReps": "3-4 sets of 8-12 reps"
    },
    {
      "Exercise": "Cat Stretch",
      "BodyPart": "Lower Back",
      "SetsReps": "3-4 sets of 8-12 reps"
    },
    {
      "Exercise": "Kneeling Forearm Stretch",
      "BodyPart": "Forearms",
      "SetsReps": "3-4 sets of 8-12 reps"
    },
    {
      "Exerc